![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 02: EDA in Healthcare
***
**Learning objectives**
- Build a binary comorbidity matrix from ICD-10 flags
- Compute pairwise co-occurrence, phi coefficients, and odds ratios
- Cluster comorbidities with hierarchical linkage and display with `clustermap`
- Identify clinically significant comorbidity pairs
- Build a network graph of comorbidity associations

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `pandas`, `numpy`, `seaborn`, `scipy`, `networkx`


## 1. Setup & Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy.cluster import hierarchy
import warnings
warnings.filterwarnings('ignore')

try:
    import networkx as nx
    print("networkx available")
except ImportError:
    print("Run: pip install networkx")

sns.set_theme(style='white', font_scale=1.02)
plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white'})

# ── Dataset ───────────────────────────────────────────────────────────────────
np.random.seed(42); N=800
sexes     = np.random.choice(['M','F'],N,p=[0.48,0.52])
payers    = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],N,p=[0.40,0.22,0.28,0.07,0.03])
ages      = np.random.normal(62,16,N).clip(18,95).astype(int)
los_days  = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
np.random.seed(99)

# Correlated comorbidity structure (realistic co-occurrence)
base = np.random.normal(size=(N, 3))   # 3 latent factors
def logistic(x): return 1/(1+np.exp(-x))

diabetes     = np.random.binomial(1, logistic(0.6*base[:,0] - 0.5), N)
hypertension = np.random.binomial(1, logistic(0.7*base[:,0] + 0.3*base[:,1] - 0.2), N)
obesity      = np.random.binomial(1, logistic(0.5*base[:,0] + 0.4*base[:,1] - 0.8), N)
ckd          = np.random.binomial(1, logistic(0.5*base[:,0] + 0.6*diabetes - 1.2), N)
heart_failure= np.random.binomial(1, logistic(0.4*base[:,1] + 0.5*hypertension - 1.5), N)
afib         = np.random.binomial(1, logistic(0.3*base[:,1] + 0.4*heart_failure - 1.8), N)
copd         = np.random.binomial(1, logistic(0.5*base[:,2] - 1.0), N)
depression   = np.random.binomial(1, logistic(0.3*base[:,2] + 0.2*base[:,0] - 1.4), N)
anxiety      = np.random.binomial(1, logistic(0.5*base[:,2] + 0.4*depression - 1.6), N)
sleep_apnea  = np.random.binomial(1, logistic(0.4*obesity + 0.3*base[:,0] - 1.5), N)

df = pd.DataFrame({
    'patient_id'   : [f'PT-{str(i).zfill(5)}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,'los_days':los_days,
    'diabetes':diabetes,'hypertension':hypertension,'obesity':obesity,
    'ckd':ckd,'heart_failure':heart_failure,'afib':afib,
    'copd':copd,'depression':depression,'anxiety':anxiety,
    'sleep_apnea':sleep_apnea,
    'readmitted_30':np.random.binomial(1,0.14,N),
})

COMORBS = ['diabetes','hypertension','obesity','ckd','heart_failure',
           'afib','copd','depression','anxiety','sleep_apnea']

print(f"Dataset: {df.shape}")
print("\nComorbidity prevalences:")
display((df[COMORBS].mean()*100).round(1).to_frame('Prevalence (%)'))


## 2. Co-occurrence Matrix

In [ ]:
# Raw co-occurrence: proportion of patients with BOTH conditions
co_matrix = pd.DataFrame(index=COMORBS, columns=COMORBS, dtype=float)
for c1 in COMORBS:
    for c2 in COMORBS:
        co_matrix.loc[c1,c2] = (df[c1] & df[c2]).mean() * 100

print("Co-occurrence matrix (% of patients with BOTH conditions):")
display(co_matrix.round(1))


## 3. Phi Coefficient Matrix (Strength of Association)

In [ ]:
def phi_coeff(a, b):
    """Phi coefficient for two binary variables.  Range: -1 to +1."""
    ct = pd.crosstab(a, b)
    if ct.shape != (2,2):
        return 0.0
    n = ct.values.sum()
    chi2 = chi2_contingency(ct)[0]
    return np.sqrt(chi2/n) * np.sign(ct.iloc[1,1]*ct.iloc[0,0] - ct.iloc[0,1]*ct.iloc[1,0])

phi_matrix = pd.DataFrame(index=COMORBS, columns=COMORBS, dtype=float)
pval_matrix= pd.DataFrame(index=COMORBS, columns=COMORBS, dtype=float)
for c1 in COMORBS:
    for c2 in COMORBS:
        phi_matrix.loc[c1,c2] = phi_coeff(df[c1], df[c2])
        if c1 != c2:
            _, p, _, _ = chi2_contingency(pd.crosstab(df[c1],df[c2]))
            pval_matrix.loc[c1,c2] = p
        else:
            pval_matrix.loc[c1,c2] = 1.0

phi_matrix = phi_matrix.astype(float)
print("Top 10 comorbidity pairs by phi coefficient:")
phi_long = (phi_matrix.where(np.triu(np.ones_like(phi_matrix,dtype=bool),k=1))
                       .stack().reset_index())
phi_long.columns = ['cond_1','cond_2','phi']
phi_long['p_value'] = [pval_matrix.loc[r.cond_1,r.cond_2] for _,r in phi_long.iterrows()]
phi_long['significant'] = phi_long['p_value'] < 0.05
display(phi_long.sort_values('phi',ascending=False).head(10).reset_index(drop=True))


## 4. Clustered Heatmap

In [ ]:
# Hierarchical clustering on phi matrix
fig = plt.figure(figsize=(10,9))
g = sns.clustermap(
    phi_matrix,
    cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
    annot=True, fmt='.2f', annot_kws={'size':8},
    linewidths=0.5, linecolor='white',
    method='ward', metric='euclidean',
    figsize=(10,9),
    cbar_kws={'label':'Phi coefficient','shrink':0.6},
    dendrogram_ratio=0.12,
)
g.ax_heatmap.set_title('Comorbidity co-occurrence — phi coefficients (Ward clustering)',
                        pad=20, fontsize=12)
g.ax_heatmap.set_xlabel(''); g.ax_heatmap.set_ylabel('')
plt.savefig('/tmp/mod02/comorbidity_clustermap.png', bbox_inches='tight')
plt.show()
print("Interpretation:")
print("  Clusters reflect shared underlying pathways (e.g. cardiometabolic, psychiatric)")


## 5. Odds Ratios for Key Pairs

In [ ]:
def odds_ratio(a, b):
    """Odds ratio and 95% CI (Woolf method) for two binary variables."""
    ct  = pd.crosstab(a, b).values
    if ct.shape != (2,2): return None, None, None
    # Apply 0.5 continuity correction if any cell is 0
    ct  = ct.astype(float) + 0.5
    OR  = (ct[1,1]*ct[0,0]) / (ct[0,1]*ct[1,0])
    log_or = np.log(OR)
    se_log  = np.sqrt(1/ct[0,0]+1/ct[0,1]+1/ct[1,0]+1/ct[1,1])
    lo  = np.exp(log_or - 1.96*se_log)
    hi  = np.exp(log_or + 1.96*se_log)
    return round(OR,2), round(lo,2), round(hi,2)

top_pairs = phi_long[phi_long['significant']].sort_values('phi',ascending=False).head(8)
rows = []
for _, r in top_pairs.iterrows():
    OR, lo, hi = odds_ratio(df[r.cond_1], df[r.cond_2])
    rows.append({'Pair':f"{r.cond_1} × {r.cond_2}",
                 'Phi':round(r.phi,3),'OR':OR,'95% CI':f'{lo}–{hi}'})
or_df = pd.DataFrame(rows)
print("Odds ratios for top comorbidity pairs:")
display(or_df)


In [ ]:
# Forest plot of odds ratios
fig, ax = plt.subplots(figsize=(9,5))
or_df_sorted = or_df.sort_values('OR')
y  = range(len(or_df_sorted))
ax.barh(y, or_df_sorted['OR'], xerr=0, color='#4878CF', height=0.5, alpha=0.8)

# Add CI lines
for i,(_, row) in enumerate(or_df_sorted.iterrows()):
    lo_val = float(str(row['95% CI']).split('–')[0])
    hi_val = float(str(row['95% CI']).split('–')[1])
    ax.plot([lo_val,hi_val],[i,i],color='#1F4E79',lw=2)
    ax.plot([lo_val,hi_val],[i,i],'|',color='#1F4E79',ms=8,mew=2)

ax.axvline(1.0,color='red',ls='--',lw=1.2,label='OR = 1 (no association)')
ax.set_yticks(list(y)); ax.set_yticklabels(or_df_sorted['Pair'],fontsize=9)
ax.set_xlabel('Odds ratio'); ax.set_title('Comorbidity pair odds ratios (95% CI)')
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig('/tmp/mod02/comorbidity_forest_plot.png', bbox_inches='tight')
plt.show()


## 6. Comorbidity Network Graph

In [ ]:
try:
    G = nx.Graph()
    G.add_nodes_from(COMORBS)

    threshold = 0.15  # minimum phi to draw an edge
    sig_pairs = phi_long[(phi_long['phi'].abs() > threshold) & phi_long['significant']]
    for _, r in sig_pairs.iterrows():
        G.add_edge(r.cond_1, r.cond_2, weight=abs(r.phi), phi=r.phi)

    pos = nx.spring_layout(G, seed=42, k=2.5)

    fig, ax = plt.subplots(figsize=(10,8))
    node_size = [df[n].mean()*5000+200 for n in G.nodes()]
    edges     = G.edges(data=True)
    widths    = [d['weight']*8 for _,__,d in edges]
    colors    = ['#D65F5F' if d['phi']>0 else '#4878CF' for _,__,d in edges]

    nx.draw_networkx_nodes(G, pos, node_size=node_size,
                           node_color='#F0E6FF', edgecolors='#6A4C93', linewidths=1.5, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, width=widths, edge_color=colors, alpha=0.7, ax=ax)

    import matplotlib.patches as mpatches
    pos_patch = mpatches.Patch(color='#D65F5F', label='Positive association')
    neg_patch = mpatches.Patch(color='#4878CF', label='Negative association')
    ax.legend(handles=[pos_patch,neg_patch], loc='lower left', fontsize=10)
    ax.set_title('Comorbidity network (edge width ∝ phi coefficient, node size ∝ prevalence)',
                 fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('/tmp/mod02/comorbidity_network.png', bbox_inches='tight')
    plt.show()
    print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges (phi > {threshold})")
except Exception as e:
    print(f"networkx not available or error: {e}")


## 7. Knowledge Check
1. Why is the phi coefficient preferred over raw co-occurrence percentages for comparing comorbidity strength?
2. Diabetes and CKD cluster together. Name the clinical mechanism driving this.
3. An OR of 2.8 for obesity × sleep apnea means what, in plain language?
4. In the network graph, what would a negative phi edge (blue) between two conditions mean?
5. Re-run the clustermap restricted to patients aged 65+. Do the clusters change?


In [ ]:
# Exercise 5 — clustermap for 65+ patients
df_65 = df[df['age'] >= 65]
phi_65 = pd.DataFrame(index=COMORBS, columns=COMORBS, dtype=float)
for c1 in COMORBS:
    for c2 in COMORBS:
        phi_65.loc[c1,c2] = phi_coeff(df_65[c1], df_65[c2])
phi_65 = phi_65.astype(float)
g65 = sns.clustermap(phi_65, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
                     annot=True, fmt='.2f', annot_kws={'size':7},
                     method='ward', figsize=(9,8))
g65.ax_heatmap.set_title('Comorbidity phi — patients 65+ only', pad=20)
plt.savefig('/tmp/mod02/comorbidity_clustermap_65plus.png', bbox_inches='tight')
plt.show()


***
**Next → NB-07: Geospatial Health Disparities Map**
